# Quantum Computing 

Famiglie DOCDB del settore *quantum computing* su PATSTAT (EPO TIP), con i concetti di ricerca del file
`epo_patent_insight_report-quantum_computing-supplementary-materials_en.xlsx` (foglio *Search concepts*).

**Settore** = CPC core (`G06N10/low`, `G06N99/002`) ∪ keyword generali ∪ keyword "realizzazioni fisiche"
(∪ keyword ampie, se `INCLUDE_BROAD_TERMS = True`). Il testo cercato è titolo + abstract.

In [2]:
from pathlib import Path

import pandas as pd
from sqlalchemy import and_, func, or_, select

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

ENV = "PROD"                 # rilanciare con "PROD" per il risultato finale
START_YEAR, END_YEAR = 1990, 2023   # anno di primo deposito (earliest_filing_year)
IPR_TYPE = "PI"              # Patented Inventions: esclude i modelli di utilità
INCLUDE_BROAD_TERMS = True   # quantum simulat+ / network+ / repeater

OUTPUT_DIR = Path("outputs/step0_epo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()
DIALECT = db.get_bind().dialect.name

Dialetto SQL del backend: bigquery


## 1. Traduzione della sintassi EPO in espressioni regolari

Il testo viene portato in minuscolo; "parola" = sequenza di `[a-z0-9]`, tutto il resto è separatore.

| Sintassi EPO | Significato | Traduzione regex |
|---|---|---|
| `a w b` | parole adiacenti, nell'ordine | `w(a, b)` |
| `a nw b` | nell'ordine, fino a n parole in mezzo | `nw(a, b, n)` |
| `a d b`, `a nD b` | qualsiasi ordine, fino a n parole in mezzo (`d` = 0) | `nd(a, b, n)` |
| `+` | troncamento, 0 o più caratteri | `T` = `[a-z0-9]*` |
| `?` | 0 o 1 carattere | `Q` = `[a-z0-9]?` |
| `_` | attaccato o separato | `J` = `[ -]?` |
| `[b,d,tr]` | alternativa di stringhe | `alt(...)` = `(b|d|tr)` |

Ogni pattern è racchiuso tra delimitatori di parola `(^|[^a-z0-9]) … ([^a-z0-9]|$)`; si usano solo costrutti
comuni a PostgreSQL, BigQuery, Spark e MySQL.

In [3]:
SEP = r"[^a-z0-9]+"   # separatore tra parole
WORD = r"[a-z0-9]+"   # una parola
T = r"[a-z0-9]*"      # EPO '+'
Q = r"[a-z0-9]?"      # EPO '?'
J = r"[ -]?"          # EPO '_'


def alt(*parts):
    """Alternativa: (p1|p2|...)."""
    return "(" + "|".join(parts) + ")"


def w(*parts):
    """EPO 'w': parole adiacenti nell'ordine dato."""
    return "(" + SEP.join(parts) + ")"


def nw(a, b, n):
    """EPO 'nw': a poi b, con al massimo n parole in mezzo."""
    return "(" + a + SEP + "(" + WORD + SEP + "){0," + str(n) + "}" + b + ")"


def nd(a, b, n):
    """EPO 'nD': a e b in qualsiasi ordine, con al massimo n parole in mezzo."""
    return alt(nw(a, b, n), nw(b, a, n))


def word(p):
    """Delimita il pattern a parole intere."""
    return "(^|[^a-z0-9])" + p + "([^a-z0-9]|$)"

## 2. Concetti di ricerca (dal foglio *Search concepts*)

In [4]:
# --- Blocchi riusati ---------------------------------------------------------
QU_BDT = "(multi" + J + ")?qu" + J + "(b|d|tr)it" + Q          # [,multi]_qu_[b,d,tr]it?
QUBIT = alt(w("quantum", "bit" + Q), QU_BDT)                      # quantum w bit?, ...

QC_CORE = alt(                                                    # lato destro del '5d' EPO
    QUBIT,
    w("quantum", "comput" + T),
    w("quantum", "gate" + Q),
    w("quantum", "circuit" + Q),
    w("quantum", "simulat" + T),
    w("quantum", "processor" + Q),
    w("quantum", "chip" + Q),
)

PHYS_TECH = alt(                                                  # lato sinistro del '5d' EPO
    "wave" + J + "guide" + Q,
    "cavit(y|ies)",
    "resonator" + Q,
    "majorana",
    "cooper" + J + "pair" + Q,
    w("josephson", "junction" + Q),
    w(alt("nitrogen", "nv"), "vacan" + T),
    w("neutral", "atom" + Q),
    w("rydberg", alt("atom" + Q, "state" + Q)),
    "divincenzo",
    "kane",
    "quantum" + J + "dot" + Q,
    "super" + J + "conduct" + T,
    "squid" + Q,
    w("paul", "trap" + Q),
    nd("trap" + T, alt("ion" + Q, "atom" + Q), 0),                # trap+ d (ion?,atom?)
)

# --- Quantum computing in general -------------------------------------------
GENERAL_KW = {
    "instruction_code": w("quantum", alt("instruction" + Q, "(op" + J + ")?code" + Q)),
    "dev_kit": w("quantum", "develop" + T, "kit" + Q),
    "sdk": w("quantum", "sdk" + Q),
    "software": w("quantum", "software" + Q),
    "language": w("quantum", "lang" + T),
    "assembler": w("quantum", alt("assembler" + Q, w("assembly", "lang" + T))),
    "qiskit": "qiskit",
    "programming": w("quantum", "program" + T),
    "kernel": w("quantum", "kernel" + Q),
    "neural_net": w("quantum", "neur" + T, "net" + T),
    "machine_learning": w("quantum", "machine", "learning"),
    "qaoa": alt(
        w("quantum", "approximat" + T, "optimi" + T, "algo" + T),
        w("quantum", "alternat" + T, "operator" + Q, "ansatz"),
    ),
    "vqe": w("variational", "quantum", "eigen" + J + "solv" + T),
    "hadamard_fourier": w("quantum", alt("hadamard", "fourier")),
    "named_algorithms": nd(alt("deutsch" + J + "jozsa", "shor", "grover") + "s?", "algo" + T, 2),
    "turing_state_machine": w("quantum", alt("turing", w("state", "machine" + Q))),
    "topological_adiabatic": alt(w("topological", "quantum"), w("adiabatic", "quantum")),
    "named_gates": alt(w(alt("clifford", "hadamard", "swap", "pauli"), "gate" + Q), "t" + J + "gate" + Q),
    "annealing": w("quantum", "anneal" + T),
    "synthesis": w("quantum", "synthesis"),
    "compiler": w("quantum", "compil" + T),
    "qram": alt(w("quantum", alt("ram" + Q, w("random", "access", "memor" + T))), "q" + J + "ram" + Q),
    "information_processing": w("quantum", "information", "processing"),
    "chip_processor": w("quantum", alt(
        "chip" + Q,
        "processor" + Q,
        w("processing", alt("unit" + Q, "device" + Q, "apparatus" + Q + Q, "system" + Q)),
    )),
    "register": w("quantum", "register" + Q),
    "bus": w("quantum", "bus(es)?"),
    "fluxmon_transmon": alt("flux" + J + "mon" + Q, "trans" + J + "mon" + Q),
    "circuit": w("quantum", "circuit" + Q),
    "gate": w("quantum", "gate" + Q),
    "computing": w("quantum", "comput" + T),
    "qubit": QUBIT,
}

GENERAL_KW_BROAD = {
    "simulation": w("quantum", "simulat" + T),
    "repeater": w("quantum", "repeater" + Q),
    "network": w("quantum", "network" + T),
}

# --- Physical realisations of quantum computing ------------------------------
PHYS_KW = {
    "coupler": w(alt("quantum", QUBIT), "coupler" + Q),
    "qubit_control": nd(QUBIT, "control" + T, 2),
    "nisq": alt("nisq", w("noisy", "intermed" + T, "scale" + T, "quantum" + T)),
    "tech_near_qc": nd(PHYS_TECH, QC_CORE, 5),
    "qubit_type": w(alt("photon" + T, "flux" + T, "charge" + T, "phase" + T), QUBIT),
}

SECTOR_KW = {**GENERAL_KW, **PHYS_KW, **(GENERAL_KW_BROAD if INCLUDE_BROAD_TERMS else {})}

# --- Classi CPC core ('/low' tradotto in prefissi) ----------------------------
CPC_CORE = ("G06N10/", "G06N99/002")

# --- Prefiltro economico (LIKE) prima delle regex -----------------------------
ANCHORS = (
    "quantum", "qubit", "qudit", "qutrit", "qu bit", "qu-bit", "qiskit",
    "transmon", "trans mon", "trans-mon", "fluxmon", "flux mon", "flux-mon",
    "nisq", "shor", "grover", "deutsch", "clifford", "pauli", "hadamard",
    "swap gate", "t gate", "t-gate", "tgate", "qram", "q-ram", "q ram",
)

## 3. Funzioni per la query
`rx()` sceglie l'operatore regex in base al dialetto SQL del backend.

In [5]:
def rx(col, pattern):
    """Condizione booleana 'col contiene pattern' nel dialetto SQL del backend."""
    d = DIALECT.lower()
    if d.startswith("postgres"):
        return col.op("~", is_comparison=True)(pattern)
    if d == "bigquery":
        return func.regexp_contains(col, pattern)
    if d in ("databricks", "spark", "sparksql", "hive"):
        return col.op("rlike", is_comparison=True)(pattern)
    if d in ("mysql", "mariadb"):
        return col.op("regexp", is_comparison=True)(pattern)
    return col.regexp_match(pattern)


def cpc_like(prefixes):
    symbol = func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "")
    return or_(*(symbol.like(p.replace(" ", "") + "%") for p in prefixes))


TXT = func.lower(func.concat(
    func.coalesce(TLS202_APPLN_TITLE.appln_title, ""),
    " ",
    func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, ""),
))

## 4. Query: famiglie del settore
Una sola query: domanda con CPC core **oppure** con almeno un match testuale (prefiltro `LIKE` + regex).

In [11]:
cpc_cond = TLS201_APPLN.appln_id.in_(
    select(TLS224_APPLN_CPC.appln_id).where(cpc_like(CPC_CORE))
)

text_cond = and_(
    or_(*(TXT.like(f"%{a}%") for a in ANCHORS)),
    or_(*(rx(TXT, word(p)) for p in SECTOR_KW.values())),
)

q = (
    db.query(TLS201_APPLN.docdb_family_id)
    .select_from(TLS201_APPLN)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(TLS201_APPLN.docdb_family_id > 0)
    .filter(TLS201_APPLN.earliest_filing_year.between(START_YEAR, END_YEAR))
    .filter(TLS201_APPLN.ipr_type == IPR_TYPE)
    .filter(or_(cpc_cond, text_cond))
    .distinct()
)

sector_ids = {row[0] for row in q.all()}
print("Famiglie nel settore:", len(sector_ids))

pd.DataFrame(sorted(sector_ids), columns=["docdb_family_id"]).to_csv(
    OUTPUT_DIR / f"qc_families_v2_{ENV}.csv", index=False
)

Famiglie nel settore: 12177


## 5. Confronto con la query v1


In [12]:
from pathlib import Path
import pandas as pd
from sqlalchemy import and_, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

ENV = "PROD"  # per il risultato finale rilanciare con "PROD"
OUTPUT_DIR = Path("outputs/step0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()

SECTOR_NAME = "Quantum Computing"

# Classi CPC "core": prefisso senza spazi
SECTOR_CPC_PREFIXES = (
    "G06N10/",      # Quantum computing (dal 2019), include /20 /40 /60 /70 /80
    "G06N99/002",   # Quantum computers (collocazione storica, pre-2019)
    "Y10S977/933",  # Spintronics or quantum computing (collezione storica)
)

# Classi CPC "adiacenti": valide solo se accompagnate da testo quantistico
ADJACENT_CPC_PREFIXES = (
    "B82Y10/00",    # Nanotecnologie per elaborazione dell'informazione
    "H10N60/",      # Dispositivi superconduttori
    "H01L39/",      # Dispositivi superconduttori (vecchia collocazione)
    "H03K19/195",   # Logica superconduttiva
)

SECTOR_KEYWORDS = (
    "quantum comput",   # computer, computing, computation
    "qubit",
    "quantum bit",
    "quantum processor",
    "quantum gate",
    "quantum circuit",
    "quantum annealing",
    "quantum error correction",
    "quantum algorithm",
    "quantum simulator",
)

# Keyword generiche: usate SOLO in AND con le classi CPC adiacenti
ADJACENT_KEYWORDS = ("quantum", "josephson")
#La giunzione Josephson è il componente fisico su cui si basano i qubit superconduttori, che sono la piattaforma hardware più diffusa nel quantum computing
#Sono due elettrodi superconduttori separati da una barriera isolante sottilissima, tipicamente alluminio, spessa circa 1–2 nm.

START_YEAR = 1990   # anno di primo deposito della famiglia
END_YEAR = 2023

IPR_TYPE = "PI"  # "PI" = Patented Inventions (esclude i modelli di utilità)


def symbol_prefix_condition(symbol_column, codes):
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


def keyword_condition(terms):
    title = func.lower(func.coalesce(TLS202_APPLN_TITLE.appln_title, ""))
    abstract = func.lower(func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, ""))
    return or_(*(col.like(f"%{t.lower()}%") for t in terms for col in (title, abstract)))


def sector_family_ids():
    family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")

    def base():
        return (
            db.query(family_id)
            .select_from(TLS201_APPLN)
            .filter(TLS201_APPLN.docdb_family_id > 0)  # esclude domande senza famiglia
            .filter(TLS201_APPLN.earliest_filing_year.between(START_YEAR, END_YEAR))
            .filter(TLS201_APPLN.ipr_type == IPR_TYPE) 
        )

    def join_cpc(q):
        return q.join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)

    def join_text(q):
        return (
            q.outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
            .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        )

    # Perimetro CPC core
    by_cpc = join_cpc(base()).filter(
        symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, SECTOR_CPC_PREFIXES)
    )

    # Perimetro keyword specifiche
    by_keywords = join_text(base()).filter(keyword_condition(SECTOR_KEYWORDS))

    # CPC adiacente AND keyword generica
    by_adjacent = join_text(join_cpc(base())).filter(
        and_(
            symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, ADJACENT_CPC_PREFIXES),
            keyword_condition(ADJACENT_KEYWORDS),
        )
    )

    return by_cpc.union(by_keywords).union(by_adjacent)


sector_v1 = sector_family_ids()

In [13]:

v1 = {row[0] for row in set(sector_v1)}
v2 = sector_ids
print(f"v1: {len(v1)}  v2: {len(v2)}  comuni: {len(v1 & v2)}  solo v1: {len(v1 - v2)}  solo v2: {len(v2 - v1)}")
pd.Series(sorted(v1 - v2), name="docdb_family_id").to_csv(OUTPUT_DIR / "solo_v1.csv", index=False)
pd.Series(sorted(v2 - v1), name="docdb_family_id").to_csv(OUTPUT_DIR / "solo_v2.csv", index=False)


v1: 13523  v2: 12177  comuni: 11293  solo v1: 2230  solo v2: 884
